# PAI 最終課題 kojikoji_s
## オフィスなどにおけるノート型パソコン搬送に向けた双腕ロボットの基礎検証

ROS 2 / Gazebo / LeRobot $\pi_{0.5}$。固定環境内で、ノート型パソコンを想定した簡略化モデルを双腕で受取台へ搬送します。移動台車や人への直接手渡しは含みません。

**最初の独立した設定セルで実行方法を選択し、その次の準備セルから8まで上から順に実行してください。** デフォルトは「Google Driveの利用可能な最新の保存データを使用」です。DriveやHugging Faceの認証があるため、各セルの完了を確認しながら進めてください。

| プルダウンの選択 | 自動で行う処理 |
|---|---|
| Google Driveの利用可能な最新の保存データを使用 | 最新の利用可能な実行フォルダを選び、両視点の成功記録があれば再利用。不足していれば保存済みモデルで再テスト。再学習はしません。 |
| 保存データを指定：実行日時 | 「Driveに接続して保存データの一覧を取得」を押して、希望の日時を選択。指定したデータで再利用・再テストします。 |
| 新規にデータ収集・学習を行う | 保存データを使用せず、データ収集・初回学習・追加学習・両視点のテストを行います。以前の`USE_GOOGLE_DRIVE = False`に相当します。 |

保存日時の指定は、保存した学習済みモデルや実行記録の選択です。GitHubの過去のプログラム版を選ぶ機能ではありません。日時はフォルダ名の値で、タイムゾーンを自動変換しません。最新を選んだ場合は、不完全な保存データを除外します。日時を指定した場合は、そのデータが利用不能なら停止します。

どちらも最後は **提出動画 → 両視点記録の確認 → カメラ画像とグラフ → 斜め前方・上方それぞれの動画 → Drive保存** です。カメラが片方しかない状態を完了扱いにせず、別試行・教師画像・生成画像で穴埋めしません。

新規学習・再評価にはGoogle ColabのA100 GPU（GPUメモリ80GB）を使用し、ハイメモリ設定を選択してください。GPUメモリとシステムRAMは別の資源です。保存済みの両視点記録だけを再出力する場合はGPU処理を行いません。Driveを入力に使わない場合でも、最後の8では成果物の保全のためDriveへ保存します。

2026年10月18日、従来の学習・制御構成のUAT完了を利用者が確認しました。今回の保存データ選択と進行管理・両視点記録を含む構成は、合成データと分岐テストで確認していますが、Colab/Gazeboでの一連のUATは未実施です。方策・学習式・双腕制御式は変更していません。追加撮影による保存負荷を含めて再評価結果を確認してください。

学習は4・5、推論は6です。方策は2視点の画像・関節状態・タスク指示から関節目標を予測し、通常の双腕協調制御が設計済み経路上の進行量へ変換します。推論中はGazeboの物理シミュレーションの進行を停止します。

本ノートブックの「評価」は、学習済みモデルで搬送に成功するかを確認するテストを意味します。カメラは真上（overhead）と斜め前方（front）の2台です。

### 接続・認証エラーが出た場合

- **`Not connected to a runtime`**：Google Driveのエラーではありません。先にColab右上の「接続」でランタイムへ接続し、その後、エラーになったセルを再実行してください。
- **Google Driveの接続・認証エラー（`mount failed`など）**：ランタイムへの接続を確認したうえで、一覧取得ボタン、またはDriveへの接続を行う準備セル・8を個別に再実行してください。認証画面が表示された場合は、保存データのあるGoogleアカウントを選んでアクセスを許可してください。既に接続済みの場合は認証画面が再表示されないことがあります。
- 再実行しても接続できない場合は、エラー内容を確認してください。セルの再実行だけで必ず解決するわけではありません。設定セルの再実行は選択をデフォルトへ戻します。準備セルの再実行は、新しい作業フォルダの作成やモデルの再コピーを伴う場合があります。


## 実行方法の選択

このセルを実行するとプルダウンが表示されます。**通常はデフォルトの「最新」を選んだまま、次の準備セルへ進んでください。** 保存日時を指定したい場合は一覧取得ボタンを押し、Google Driveへのアクセスを許可してから希望のフォルダを選択します。新規に学習する場合は、プルダウンの「新規にデータ収集・学習を行う」を選びます。True/Falseを手入力する必要はありません。

選択を変更したら、直後の準備セルを実行して確定してください。準備後にプルダウンだけ変更しても、実行中の作業には反映されません。


In [ ]:
# 実行方法の選択
from pathlib import Path
import datetime, re
import ipywidgets as widgets
from IPython.display import display

# 最初は「最新」を選択。新規実行ならDriveの読み込みをせずに選べます。
run_selection=widgets.Dropdown(
    options=[('Google Driveの利用可能な最新の保存データを使用','latest'),
             ('新規にデータ収集・学習を行う（保存データを使用しない）','new')],
    value='latest',description='実行方法:',layout=widgets.Layout(width='100%'))
load_saved_button=widgets.Button(description='Driveに接続して保存データの一覧を取得',layout=widgets.Layout(width='340px'))
selection_status=widgets.HTML(value='最新の保存データを使用する設定です。そのまま次のセルへ進めます。日時を指定する場合は一覧を取得してください。')

def refresh_saved_choices(_):
    load_saved_button.disabled=True
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        candidates=[]
        for base in [Path('/content/drive/MyDrive/PAI_results'),Path('/content/drive/MyDrive/PAI_backup')]:
            if not base.is_dir():continue
            for folder in base.iterdir():
                if not folder.is_dir():continue
                match=re.match(r'^(\d{8}_\d{6})(?:$|_)',folder.name)
                if not match:continue
                try:date=datetime.datetime.strptime(match.group(1),'%Y%m%d_%H%M%S')
                except ValueError:continue
                candidates.append((date,str(folder),folder))
        options=[('Google Driveの利用可能な最新の保存データを使用','latest'),
                 ('新規にデータ収集・学習を行う（保存データを使用しない）','new')]
        options += [('保存データを指定：'+f.name+'（'+f.parent.name+'）',str(f))
                    for _,_,f in sorted(candidates,reverse=True)]
        previous=run_selection.value
        run_selection.options=options
        run_selection.value=previous if previous in [value for _,value in options] else 'latest'
        selection_status.value=f'保存フォルダを{len(candidates)}件表示しました。利用可能かどうかは次のセルで確認します。'
    except Exception as exc:
        import html
        selection_status.value=html.escape('一覧取得に失敗しました。接続・認証を確認してボタンを押し直してください。'+type(exc).__name__+': '+str(exc))
    finally:load_saved_button.disabled=False

def selection_changed(change):
    import html
    choice=change['new']
    message=('利用可能な最新の保存データを自動選択します。' if choice=='latest'
             else '保存データを使わず、新規にデータ収集・学習を行います。' if choice=='new'
             else '指定した保存フォルダを使用します：'+choice)
    selection_status.value=html.escape(message+' 選択後、次のセルを実行してください。')
run_selection.observe(selection_changed,names='value')
load_saved_button.on_click(refresh_saved_choices)
display(run_selection,load_saved_button,selection_status)


## 選択した方法で作業環境を準備

設定セルで選んだ内容を使用します。準備完了後は1から8まで順に実行してください。


In [ ]:
# 上の設定セルの選択を確定し、作業環境を準備します。
if 'run_selection' not in globals():
    raise RuntimeError('先に「実行方法の選択」のセルを実行してください。')
SELECTED_RUN_OPTION = run_selection.value
USE_GOOGLE_DRIVE = SELECTED_RUN_OPTION != 'new'

from pathlib import Path
import os, sys, subprocess, json, datetime, getpass, hashlib, shutil, re
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('Google ColabのLinuxランタイムを使用してください。新規学習・再評価にはA100 GPUが必要です。')

def valid_checkpoint(folder):
    try:
        r=json.loads((folder/'training_result.json').read_text())
        return (r.get('status')=='trained' and r.get('task_training_completed') is True
                and r.get('task')=='it_laptop_handover_fixed_cell' and r.get('steps_completed',0)>=1000
                and r.get('base_revision')=='22c30e5b34f9e7ab738b1b34fc2fc5ecfe2d28f0'
                and (folder/'expert.safetensors').stat().st_size>0)
    except (OSError,ValueError,TypeError):return False

def both_views_complete(evaluation):
    try:
        summary=json.loads((evaluation/'summary.json').read_text())
        if summary.get('hybrid_successes')!=2 or summary.get('hybrid_attempts')!=2:return False
        for seed in [900,901]:
            folder=evaluation/f'hybrid_{seed}';r=json.loads((folder/'result.json').read_text())
            if r.get('mode')!='policy' or not r.get('completed') or not r.get('success'):return False
            trace=json.loads((folder/'control_trace.json').read_text())
            if len(trace)<2:return False
            for name in ['front','overhead']:
                base=folder/'camera_views'/name;frames=json.loads((base/'frames.json').read_text())
                if len(frames)<2:return False
                times=[float(row[0]) for row in frames]
                if any(not __import__('math').isfinite(t) for t in times):return False
                if any(b<=a for a,b in zip(times,times[1:])):return False
                if times[0]>float(trace[0]['sim_t'])+.5 or times[-1]<float(trace[-1]['sim_t'])-.5:return False
                if not all((base/f'{int(i):05d}.jpg').is_file() for _,i in frames):return False
        return True
    except (OSError,ValueError,TypeError,KeyError,IndexError):return False

def latest_saved_source(roots):
    candidates=[]
    for root in roots:
        if not root.is_dir():continue
        for folder in root.iterdir():
            if not folder.is_dir():continue
            match=re.match(r'^(\d{8}_\d{6})(?:$|_)',folder.name)
            if not match:continue
            try:date=datetime.datetime.strptime(match.group(1),'%Y%m%d_%H%M%S')
            except ValueError:continue
            candidates.append((date,str(folder),folder))
    for _,_,folder in sorted(candidates,reverse=True):
        checkpoints=sorted([p for p in folder.glob('checkpoint*') if valid_checkpoint(p)],key=lambda p:(p.name=='checkpoint_refined',p.name),reverse=True)
        evaluations=sorted({p.parent for p in folder.glob('evaluation*/evaluation/summary.json')},reverse=True)
        # 同じ実行フォルダ内の時刻名は日付を含まないことがあるため、再利用対象の名前を表示します。
        complete=next((p for p in evaluations if both_views_complete(p)),None)
        if checkpoints or complete:return folder,(checkpoints[0] if checkpoints else None),complete
        print('利用可能なモデル・両視点記録がないため除外:',folder.name,flush=True)
    raise RuntimeError('利用可能な保存済みモデル・両視点評価がありません。Driveの保存状況を確認してください。自動で新規学習へ切り替えません。')

SOURCE_RUN=None;SOURCE_CHECKPOINT=None;SOURCE_EVALUATION=None
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    if SELECTED_RUN_OPTION == 'latest':
        SOURCE_RUN,SOURCE_CHECKPOINT,SOURCE_EVALUATION=latest_saved_source([
            Path('/content/drive/MyDrive/PAI_results'),Path('/content/drive/MyDrive/PAI_backup')])
    else:
        SOURCE_RUN=Path(SELECTED_RUN_OPTION)
        if not SOURCE_RUN.is_dir():raise RuntimeError('指定した保存フォルダがありません: '+str(SOURCE_RUN))
        checkpoints=sorted([p for p in SOURCE_RUN.glob('checkpoint*') if valid_checkpoint(p)],
                           key=lambda p:(p.name=='checkpoint_refined',p.name),reverse=True)
        evaluations=sorted({p.parent for p in SOURCE_RUN.glob('evaluation*/evaluation/summary.json')},reverse=True)
        SOURCE_CHECKPOINT=checkpoints[0] if checkpoints else None
        SOURCE_EVALUATION=next((p for p in evaluations if both_views_complete(p)),None)
        if SOURCE_CHECKPOINT is None and SOURCE_EVALUATION is None:
            raise RuntimeError('指定した保存データには、利用可能な学習済みモデル・両視点評価がありません。別の保存日時を選んでください。自動で別の版や新規学習へ切り替えません。')
    print('選択した保存データ:',SOURCE_RUN)
NEED_TRAINING=not USE_GOOGLE_DRIVE
NEED_EVALUATION=SOURCE_EVALUATION is None
run_id=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN=Path('/content/pai_runs')/run_id;RUN.mkdir(parents=True,exist_ok=False)
ROOT=Path('/content')/('pai-laptop-handover_'+run_id)
subprocess.run(['git','clone','https://github.com/1976koji/pai-laptop-handover.git',str(ROOT)],check=True)
AI_PYTHON=Path('/content/pai_kojikoji_s/venv_cu128_v4/bin/python')
CHECKPOINT=RUN/'checkpoint_refined'
def run(args):
    env=os.environ.copy()
    for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):env.pop(key,None)
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True,env=env)
if NEED_EVALUATION:run(['nvidia-smi'])
if SOURCE_CHECKPOINT is not None:
    CHECKPOINT.mkdir()
    for name in ['training_result.json','expert.safetensors']:
        print('復元中:',name,flush=True)
        shutil.copy2(SOURCE_CHECKPOINT/name,CHECKPOINT/name)
        if (CHECKPOINT/name).stat().st_size!=(SOURCE_CHECKPOINT/name).stat().st_size:
            raise RuntimeError('復元ファイルのサイズ不一致: '+name)
if SOURCE_EVALUATION is not None:
    EVAL=RUN/'evaluation_restored';EVAL.mkdir()
    print('両視点の評価記録を復元中:',SOURCE_EVALUATION,flush=True)
    shutil.copytree(SOURCE_EVALUATION,EVAL/'evaluation')
else:EVAL=RUN/('evaluation_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
(RUN/'run_origin.json').write_text(json.dumps(dict(use_google_drive=USE_GOOGLE_DRIVE,selected_run_option=SELECTED_RUN_OPTION,source_run=str(SOURCE_RUN) if SOURCE_RUN else None,
    source_evaluation=str(SOURCE_EVALUATION) if SOURCE_EVALUATION else None,needs_training=NEED_TRAINING,needs_evaluation=NEED_EVALUATION),indent=2))
print('作業先:',RUN)
print('自動判定:', '新規学習と両視点評価' if NEED_TRAINING else '保存済みモデルで両視点を再評価' if NEED_EVALUATION else '保存済みの両視点評価を再利用')
print('以降は上から順に実行してください。不要な処理は自動でスキップします。')


## 1 環境構築
APT導入には時間がかかります。ロボット資産は固定revisionから取得します。
Driveを使う場合の不要な処理は自動でスキップします。セルの順序を変える必要はありません。


In [ ]:
if NEED_EVALUATION:
    run(['bash','it_desk/setup_sim.sh'])
    run([sys.executable,'it_desk/setup_ai.py','--venv',AI_PYTHON.parent.parent])
    run(['bash','scripts/ros_run.sh','prepare_robot.py'])
    run(['bash','scripts/ros_run.sh','verify_static.py'])
    run(['bash','scripts/ros_run.sh','test_control_interface.py'])

else:
    print('保存済み両視点評価を使用するため、ROS/GPU環境構築をスキップします。')

if shutil.which('ffmpeg') is None:
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','-qq','ffmpeg'])


## 2 Hugging Face認証処理の定義

このセルはそのまま実行してください。ここでのトークン入力は不要です。

学習・評価時に、利用者本人のColab Secretsから`HF_TOKEN`を取得します。取得できない場合のみ、非表示の入力欄が表示されます。処理が成功したトークンは同じセッションのメモリ内で再利用し、ファイルやカーネルの環境変数には保存しません。セッションの終了・再起動、またはこのセルの再実行で保持は解除されます。


In [ ]:
# 同一セッション内だけで、成功した処理のトークンを再利用します。
for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
    os.environ.pop(key,None)
_hf_token_cache=None

def clear_hf_token():
    global _hf_token_cache
    _hf_token_cache=None

def require_trained(folder, steps):
    folder=Path(folder)
    report_path=folder/'training_result.json'
    weights=folder/'expert.safetensors'
    if not report_path.is_file() or not weights.is_file():
        raise RuntimeError('学習完了ファイルがありません。前の学習セルを完了してください。')
    report=json.loads(report_path.read_text())
    if (report.get('status')!='trained'
        or report.get('task_training_completed') is not True
        or report.get('task')!='it_laptop_handover_fixed_cell'
        or report.get('steps_completed',0)<steps
        or weights.stat().st_size==0):
        raise RuntimeError('前の学習が未完了または失敗しています。トークン入力と次の処理は行いません。')
    return report

def run_ai(args, completed_check=None):
    global _hf_token_cache
    env=os.environ.copy()
    for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
        env.pop(key,None)
    token=None
    try:
        token=_hf_token_cache
        if not token:
            try:
                from google.colab import userdata
                token=userdata.get('HF_TOKEN')
            except Exception:
                token=None
            if not token:
                token=getpass.getpass('Hugging Face読み取りトークン（非表示）: ')
        if not token:raise RuntimeError('トークンが空です。')
        env['HF_TOKEN']=token
        subprocess.run([str(x) for x in args],cwd=ROOT,check=True,env=env)
        if completed_check is not None:
            completed_check()
        _hf_token_cache=token
    except BaseException:
        clear_hf_token()
        raise
    finally:
        env.pop('HF_TOKEN',None)
        token=None

def train_stage(folder, steps, args):
    # 完了済みなら再学習しません。不完全な既存重みは上書きしません。
    folder=Path(folder)
    if (folder/'expert.safetensors').exists():
        require_trained(folder,steps)
        print('学習完了済み。既存モデルを使用します。')
        return
    run_ai(args,completed_check=lambda:require_trained(folder,steps))
    print('学習完了を確認しました。次のセルへ進めます。')


## 3 教師データ収集
失敗を含めて保存します。

`it_desk/collect.py`で教師制御の実行データを収集します。この段階ではモデルの重みは更新しません。

Driveを使う場合の不要な処理は自動でスキップします。セルの順序を変える必要はありません。


In [ ]:
if NEED_TRAINING:
    run(['bash','scripts/ros_run.sh','collect.py','--out',RUN/'train','--count','24','--seed','100','--duration','150'])
    run(['bash','scripts/ros_run.sh','collect.py','--out',RUN/'validation','--count','6','--seed','300','--duration','150'])
    for split,minimum in [('train',12),('validation',4)]:
        s=json.loads((RUN/split/'summary.json').read_text())
        print(split,'usable:',s['usable_successes'])
        if s['usable_successes']<minimum:raise RuntimeError('成功データ不足。失敗の原因を確認してください。')

else:
    print('Driveの保存データを使用するため、この学習関連処理をスキップします。')


## 4 初回学習 500ステップ
3の成功データ数を確認してから実行してください。

`it_desk/train.py`で、教師データから$\pi_{0.5}$の動作生成部分を500ステップ追加学習します。重みは`RUN/checkpoint/expert.safetensors`に保存します。

Driveを使う場合の不要な処理は自動でスキップします。セルの順序を変える必要はありません。


In [ ]:
if NEED_TRAINING:
    train_stage(RUN/'checkpoint',500,
        [AI_PYTHON,'it_desk/train.py','--train',RUN/'train','--validation',RUN/'validation','--out',RUN/'checkpoint','--steps','500'])

else:
    print('Driveの保存データを使用するため、この学習関連処理をスキップします。')


## 5 追加学習 1000ステップ

初回学習の完了を確認してから実行します。失敗・未完了の場合はトークンを求めず停止します。同じセッションでは初回学習で成功したトークンを再利用します。

`it_desk/train_refine.py`で初回学習の重みを引き継ぎ、さらに1,000ステップ学習します。重みは`RUN/checkpoint_refined/expert.safetensors`に保存します。

Driveを使う場合の不要な処理は自動でスキップします。セルの順序を変える必要はありません。


In [ ]:
if NEED_TRAINING:
    require_trained(RUN/'checkpoint',500)
    train_stage(RUN/'checkpoint_refined',1000,
        [AI_PYTHON,'it_desk/train_refine.py','--train',RUN/'train','--validation',RUN/'validation','--init',RUN/'checkpoint','--out',RUN/'checkpoint_refined','--steps','1000'])
    CHECKPOINT=RUN/'checkpoint_refined'

else:
    print('Driveの保存データを使用するため、この学習関連処理をスキップします。')


## 6 両視点を記録して学習済みモデルをテスト（評価）

保存済みの両視点評価が揃っていれば再利用し、足りなければ追加学習済みモデルで評価します。新規学習の場合も同じ評価を行います。評価はseed 900/developer、901/standardの2試行です。開発時にも使用した条件であり、未知配置での成功率ではありません。両視点の画像は同じ試行から保存しますが、カメラは非同期です。

このセルでは学習済みモデルで搬送を評価し、モデルの重みは更新しません。


In [ ]:
if NEED_EVALUATION:
    require_trained(CHECKPOINT,1000)
    if 'PAI_RECORD_CAMERA_VIEWS' not in (ROOT/'it_desk/episode.py').read_text():
        raise RuntimeError('両視点の記録に対応していないコードです。初期化した作業先を確認してください。')
    previous_recording=os.environ.get('PAI_RECORD_CAMERA_VIEWS')
    try:
        os.environ['PAI_RECORD_CAMERA_VIEWS']='1'
        run_ai([sys.executable,'scripts/evaluate.py','--python',AI_PYTHON,'--checkpoint',CHECKPOINT,'--out',EVAL])
    finally:
        if previous_recording is None:os.environ.pop('PAI_RECORD_CAMERA_VIEWS',None)
        else:os.environ['PAI_RECORD_CAMERA_VIEWS']=previous_recording
else:
    print('保存済みの両視点評価を再利用:',EVAL)
if not both_views_complete(EVAL/'evaluation'):
    raise RuntimeError('両視点・2試行の成功記録が揃っていません。評価ログとカメラ記録を確認してください。')
print('斜め前方・上方のカメラ記録と、評価2/2成功を確認しました。')


## 7 評価動画の出力・確認

Google Driveに保存したデータを使用する場合も、新規にデータ収集・学習を行う場合も、このセルを実行してください。今回選択・復元した評価から`review_2x.mp4`を出力します。シミュレーション時間の2倍速です。提出前に映像・成功結果・1〜3分の長さを確認してください。


In [ ]:
run([sys.executable,'scripts/make_review.py','--evaluation',EVAL/'evaluation','--out',EVAL/'review_2x.mp4'])
from IPython.display import Video,display
display(Video(str(EVAL/'review_2x.mp4'),embed=True))


## 7.1 両視点記録と対象試行の確認

最初のセルでGoogle Driveの保存済みデータを使用すると指定した場合は、復元した評価記録、または保存済みモデルで今回再評価した記録を確認します。新規にデータ収集・学習を行うと指定した場合は、今回学習したモデルの評価記録を確認します。いずれの場合も6で確定した同じ2試行を使用するため、このセルでデータを選び直す必要はありません。続いて7.2、7.3、8を順に実行してください。


In [ ]:
VIS_RUN=RUN
if not both_views_complete(EVAL/'evaluation'):
    raise RuntimeError('両視点記録が不足しています。6の結果を確認してください。')
for seed in [900,901]:
    folder=EVAL/'evaluation'/f'hybrid_{seed}'
    counts={view:len(json.loads((folder/'camera_views'/view/'frames.json').read_text())) for view in ['front','overhead']}
    print(f'seed {seed}: front={counts["front"]}枚 / overhead={counts["overhead"]}枚')
print('表示対象:',EVAL/'evaluation')


## 7.2 カメラ画像とロボット挙動の可視化

上部の2視点画像と5つのグラフを1画面に表示します。スライダーで選んだ画像時刻を赤線で示します。試行と関節は表示の切り替えであり、処理の実行パターンを変更するものではありません。PNGは日時・試行・画像番号・識別子を含む名前でダウンロードできます。

グラフは画像の平均絶対差分、方策由来の進行係数、関節目標・指令・実測角度、関節変化速度、搬送物と左右支持板の高さです。画像差分は後処理指標で、モデル内部の特徴や因果的寄与ではありません。物体高さはシミュレータ、支持板位置は順運動学から得ています。評価のfront JPEGでは字幕行を除いて差分を計算します。

上方画像はfront時刻以前の直近0.25秒以内の観測を対応づけます。厳密な同期撮影ではなく、対応画像がない時刻にはその旨を表示します。次の7.3では両視点の動画を自動で出力します。


In [ ]:
# 読み取り専用の後処理。学習・評価・ROS制御は実行しません。
from pathlib import Path
import json, datetime, csv, subprocess, shutil, io, base64, html, uuid
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display, clear_output, Video

# 再接続後は、保存済みの実行フォルダを明示してください（自動で最新を選びません）。
VIS_RUN = (Path(VIS_RUN) if globals().get('VIS_RUN') is not None
           else Path(RUN) if 'RUN' in globals() else None)
if VIS_RUN is None or not VIS_RUN.is_dir():
    raise RuntimeError('先に7.1で保存済みデータを選択してください。再学習は不要です。')
episodes = [EVAL/'evaluation'/f'hybrid_{seed}' for seed in [900,901]]
if not episodes:
    raise RuntimeError('control_trace.jsonがありません。評価・教師試行の保存先を確認してください。')
VIS_OUT = VIS_RUN/'visualization'/datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
VIS_OUT.mkdir(parents=True, exist_ok=False)
print('可視化出力:', VIS_OUT)
print('時刻はシミュレーション時間。画像差分は後処理の参考指標で、モデル内部特徴や因果的寄与ではありません。')

def load_visual_episode(folder):
    trace = json.loads((folder/'control_trace.json').read_text())
    if len(trace) < 2:
        raise ValueError('観測記録が2点未満です。')
    t = np.array([r['sim_t'] for r in trace], dtype=float)
    if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('制御時刻が有限かつ単調増加ではありません。')
    result = json.loads((folder/'result.json').read_text()) if (folder/'result.json').exists() else {}
    data = dict(folder=folder, trace=trace, t=t, result=result)
    npz = folder/'episode.npz'
    if npz.is_file() and result.get('mode') == 'teacher':
        with np.load(npz, allow_pickle=False) as a:
            data.update(it=a['time'].copy(), front=a['top'].copy(), overhead=a['side'].copy())
        # 既存のキー名に注意: top=front、side=overhead（episode.pyの格納順）。
        data['kind'] = 'teacher: stored 224x224 RGB observations'
        data['frame'] = lambda i: (data['front'][i], data['overhead'][i])
    else:
        index = json.loads((folder/'frames.json').read_text())
        valid = [(float(ti), folder/'frames'/f'{int(fi):05d}.jpg') for ti,fi in index
                 if (folder/'frames'/f'{int(fi):05d}.jpg').is_file()]
        if len(valid) < 2:
            raise ValueError('保存された斜め前方の画像が2枚未満です。')
        data['it'] = np.array([v[0] for v in valid]); data['paths'] = [v[1] for v in valid]
        def frame(i):
            with Image.open(data['paths'][i]) as im:
                return np.array(im.convert('RGB')), None
        data['frame'] = frame
        data['kind'] = 'recorded front JPEG; overhead not saved'
        overhead_index=folder/'camera_views'/'overhead'/'frames.json'
        if overhead_index.is_file():
            oi=json.loads(overhead_index.read_text())
            pairs=[(float(ti),overhead_index.parent/f'{int(fi):05d}.jpg') for ti,fi in oi if (overhead_index.parent/f'{int(fi):05d}.jpg').is_file()]
            if pairs:
                ot=np.array([x[0] for x in pairs]);op=[x[1] for x in pairs];front_reader=data['frame']
                def combined(i):
                    front,_=front_reader(i);ti=data['it'][i];j=int(np.searchsorted(ot,ti,side='right')-1)
                    overhead=None
                    if j>=0 and ti-ot[j]<=.25:
                        with Image.open(op[j]) as im:overhead=np.array(im.convert('RGB'))
                    return front,overhead
                data['frame']=combined;data['kind']='front JPEG + recorded overhead; latest overhead within 0.25 s'

    it = np.asarray(data['it'], dtype=float)
    if len(it) < 2 or not np.isfinite(it).all() or np.any(np.diff(it) <= 0):
        raise ValueError('画像時刻が不足、重複、または非単調です。')
    diff = np.full((len(it), 2), np.nan)
    previous = [None, None]
    for i in range(len(it)):
        for j, image in enumerate(data['frame'](i)):
            if image is None: continue
            # 評価映像は説明字幕の行を除外。生の方策入力画像とは同一ではありません。
            if 'paths' in data and j == 0: image = image[50:, :, :]
            pixels = np.asarray(Image.fromarray(image).resize((224,224)), dtype=np.float32)
            if previous[j] is not None: diff[i,j] = np.mean(np.abs(pixels-previous[j]))/255.
            previous[j] = pixels
    data['image_change'] = diff
    data['q'] = np.array([r['q'] for r in trace], dtype=float)
    data['cmd'] = np.array([r['command'] for r in trace], dtype=float)
    if data['q'].shape != (len(t),14) or data['cmd'].shape != (len(t),14):
        raise ValueError('14関節の記録形式ではありません。')
    data['pred'] = np.array([(r.get('projection') or {}).get('learned_joint_goal',[np.nan]*14) for r in trace])
    data['alpha'] = np.array([(r.get('projection') or {}).get('progress_fraction',np.nan) for r in trace])
    data['speed'] = np.r_[np.nan, np.linalg.norm(np.diff(data['q'],axis=0),axis=1)/np.diff(t)]
    return data

def save_tables(data, out):
    out.mkdir(parents=True, exist_ok=True)
    # 画像と制御は元の時刻のまま別CSVへ保存。補間値を実測値と混同しない。
    with (out/'image_metrics.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['image_sim_t_s','front_mean_abs_pixel_change_0_1','overhead_mean_abs_pixel_change_0_1'])
        w.writerows(zip(data['it'],*data['image_change'].T))
    with (out/'control_metrics.csv').open('w',newline='') as f:
        w=csv.writer(f);w.writerow(['control_sim_t_s','phase','alpha','joint_speed_norm_rad_per_s','case_x_m','case_y_m','case_z_m']+
            [f'{kind}_{side}{i}_rad' for kind in ['observed','command','learned_goal'] for side in ['L','R'] for i in range(1,8)])
        for i,r in enumerate(data['trace']):
            w.writerow([data['t'][i],r['phase'],data['alpha'][i],data['speed'][i],*r['case'],*data['q'][i],*data['cmd'][i],*data['pred'][i]])
    (out/'README.txt').write_text(
        'Postprocessing only; not a new robot evaluation.\n'+data['kind']+'\n'+
        'Camera snapshots are asynchronous latest observations, not hardware-synchronized frames.\n'
        'Pixel change = mean(abs(current-previous))/255; not learned features or causal attribution.\n'
        'Recorded front JPEG metrics exclude the top 50 caption rows; they are not raw policy inputs.\n'
        'Case/support positions come from simulator state/kinematics, not image recognition.\n'
        'Command and learned_goal are targets, not measured motion. Teacher trials have no policy alpha.\n'
        'Video uses a uniform 5 fps simulation-time grid with latest available frames (sample-and-hold).\n'
        'No timestamps for individual raw policy input images are available in evaluation logs.\n',encoding='utf-8')

def make_plot(data, joint, image_index=0):
    # カメラ2視点と5グラフを単一の画面に配置する。
    t=data['t'];tr=data['trace'];stamp=float(data['it'][image_index])
    fig=plt.figure(figsize=(12,8),layout='constrained')
    grid=fig.add_gridspec(4,2,height_ratios=[1.5,1,1,1])
    for j,(name,image) in enumerate(zip(['front','overhead'],data['frame'](image_index))):
        ax=fig.add_subplot(grid[0,j]);ax.set_axis_off()
        if image is None:
            ax.text(.5,.5,'OVERHEAD NOT RECORDED\nThis evaluation saved front only',ha='center',va='center',transform=ax.transAxes)
        else:ax.imshow(image)
        ax.set_title(name+' | sim t='+f'{stamp:.3f}s',fontsize=10)
    axes=[fig.add_subplot(grid[1,0]),fig.add_subplot(grid[1,1]),fig.add_subplot(grid[2,0]),fig.add_subplot(grid[2,1]),fig.add_subplot(grid[3,:])]
    for j,name in enumerate(['front','overhead']):
        if np.isfinite(data['image_change'][:,j]).any():axes[0].plot(data['it'],data['image_change'][:,j],label=name)
    axes[0].set_ylabel('Pixel change [0-1]');axes[0].legend(fontsize=7)
    if np.isfinite(data['alpha']).any():axes[1].plot(t,data['alpha'])
    else:axes[1].text(.05,.5,'Teacher: no policy alpha',transform=axes[1].transAxes,fontsize=9)
    axes[1].set_ylabel('Progress alpha');axes[1].set_ylim(-.05,1.05)
    for key,label in [('q','observed'),('cmd','command'),('pred','learned goal')]:
        if np.isfinite(data[key][:,joint]).any():axes[2].plot(t,data[key][:,joint],label=label)
    axes[2].set_ylabel(f"{'L' if joint<7 else 'R'}{joint%7+1} [rad]");axes[2].legend(fontsize=7,ncol=3)
    axes[3].plot(t,data['speed']);axes[3].set_ylabel('Joint speed [rad/s]');axes[3].set_ylim(0,max(.01,float(np.nanmax(data['speed']))*1.15))
    for key,label in [('case','laptop (simulator)'),('left_support','left support (FK)'),('right_support','right support (FK)')]:
        axes[4].plot(t,[r[key][2] for r in tr],label=label)
    axes[4].set_ylabel('Height [m]');axes[4].legend(fontsize=7,ncol=3)
    for ax in axes:
        ax.axvline(stamp,color='crimson',linestyle='--',linewidth=1)
        ax.set_xlabel('Simulation time [s]',fontsize=8);ax.tick_params(labelsize=8);ax.grid(alpha=.2)
    fig.suptitle(f"{data['folder'].name} | mode={data['result'].get('mode','unknown')} | success={data['result'].get('success','unknown')}",fontsize=11)
    return fig,axes

def camera_panel(data, i):
    canvas=Image.new('RGB',(900,320),(20,25,35));draw=ImageDraw.Draw(canvas)
    for k,(name,im) in enumerate(zip(['front','overhead'],data['frame'](i))):
        draw.text((15+450*k,10),name,fill='white')
        if im is None:draw.text((465,130),'Not recorded for this evaluation',fill='white')
        else:
            view=Image.fromarray(im);view.thumbnail((430,245));canvas.paste(view,(15+450*k,35))
    label=f"mode={data['result'].get('mode','unknown')} success={data['result'].get('success','unknown')} | sim t={data['it'][i]:.3f}s"
    draw.text((15,285),label,fill='white');draw.text((15,303),'Camera record / 1x simulation time (resampled at 5 fps)',fill='white')
    return canvas

def export_camera_video(data, out):
    ffmpeg=shutil.which('ffmpeg')
    if not ffmpeg: raise RuntimeError('ffmpegがありません。環境構築済みのColabで実行してください。')
    unique=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:8]
    target=out/f'{VIS_RUN.name}_{out.name}_camera_views_{unique}_1x.mp4';tmp=target.with_name(target.stem+'_building.mp4')
    times=np.arange(data['it'][0],data['it'][-1]+1e-8,.2)
    log=out/'camera_encode.log'
    with log.open('w') as err:
        p=subprocess.Popen([ffmpeg,'-y','-loglevel','error','-f','rawvideo','-pix_fmt','rgb24','-s','900x320','-r','5','-i','-',
            '-an','-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(tmp)],stdin=subprocess.PIPE,stderr=err)
        try:
            for t in times:
                i=int(np.clip(np.searchsorted(data['it'],t,side='right')-1,0,len(data['it'])-1))
                p.stdin.write(np.asarray(camera_panel(data,i),dtype=np.uint8).tobytes())
            p.stdin.close();rc=p.wait()
            if rc:raise RuntimeError('映像出力失敗: '+str(log))
        finally:
            if p.poll() is None:p.terminate();p.wait()
    tmp.replace(target);return target

# 一試行ずつ選択。異なる試行の画像と動作を混ぜません。
options=[(str(p.relative_to(VIS_RUN)),p) for p in episodes]
default=next((p for p in episodes if p.name=='hybrid_900'),episodes[0])
choice=widgets.Dropdown(options=options,value=default,description='試行:',layout=widgets.Layout(width='90%'))
joint_choice=widgets.Dropdown(options=[(f'{s}{i+1}',j*7+i) for j,s in enumerate(['L','R']) for i in range(7)],description='関節:')
load_button=widgets.Button(description='記録を読み込み・グラフ保存')
output=widgets.Output();state={}

def show_episode(_):
    load_button.disabled=True
    with output:
        clear_output(wait=True)
        try:
            data=load_visual_episode(choice.value)
            out=VIS_OUT/('_'.join(choice.value.relative_to(VIS_RUN).parts))
            save_tables(data,out);state.clear();state.update(data=data,out=out)
            print(data['kind']);print('保存先:',out)
            print('赤線＝選択画像の時刻。未記録の上方画像は補完しません。')
            slider=widgets.IntSlider(min=0,max=len(data['it'])-1,value=0,description='画像:',continuous_update=False,layout=widgets.Layout(width='60%'))
            expand=widgets.ToggleButton(value=False,description='拡大表示',icon='search-plus')
            dashboard=widgets.HTML(layout=widgets.Layout(width='100%'))
            status=widgets.HTML()
            def show_frame(change=None):
                i=slider.value
                fig,_=make_plot(data,joint_choice.value,i)
                try:
                    buf=io.BytesIO();fig.savefig(buf,format='png',dpi=120)
                    png=buf.getvalue();(out/'image_and_motion.png').write_bytes(png)
                finally:plt.close(fig)
                for name,image in zip(['front','overhead'],data['frame'](i)):
                    if image is not None:Image.fromarray(image).save(out/(name+'_selected.png'))
                camera_panel(data,i).save(out/'selected_camera_frame.png')
                width='1400px' if expand.value else '900px'
                encoded=base64.b64encode(png).decode('ascii')
                # 標準表示は画面幅に収める。拡大時だけ横スクロールを許可する。
                style=('width:1400px;max-width:none;' if expand.value else 'width:900px;max-width:100%;')
                dashboard.value='<div style="overflow-x:auto"><img alt="Camera views and synchronized robot motion plots" style="'+style+'height:auto" src="data:image/png;base64,'+encoded+'"></div>'
                nearest=int(np.argmin(abs(data['t']-data['it'][i])))
                status.value=html.escape(f"画像 {data['it'][i]:.3f}s / 最近傍制御 {data['t'][nearest]:.3f}s / 差 {abs(data['t'][nearest]-data['it'][i]):.3f}s / 工程 {data['trace'][nearest]['phase']}")
            slider.observe(show_frame,names='value');expand.observe(show_frame,names='value')
            display(widgets.HBox([slider,expand]),status,dashboard);show_frame()
            download=widgets.Button(description='表示画像PNGをダウンロード',icon='download',layout=widgets.Layout(width='250px'))
            feedback=widgets.Output()
            def download_png(_):
                with feedback:
                    clear_output(wait=True)
                    try:
                        from google.colab import files
                        unique=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:8]
                        name=f'{VIS_RUN.name}_{out.name}_frame{slider.value:05d}_{unique}.png'
                        target=out/name
                        shutil.copyfile(out/'image_and_motion.png',target)
                        files.download(str(target))
                    except ImportError:print('保存済みPNG:',out/'image_and_motion.png')
            download.on_click(download_png);display(download,feedback)
            video_button=widgets.Button(description='カメラ映像MP4を保存')
            video_output=widgets.Output()
            def save_video(_):
                video_button.disabled=True
                with video_output:
                    clear_output(wait=True)
                    try:
                        path=export_camera_video(data,out);print('保存:',path);display(Video(str(path),embed=True,width=800))
                    except Exception as exc:print(type(exc).__name__+': '+str(exc))
                    finally:video_button.disabled=False
            video_button.on_click(save_video)
            extra=widgets.Accordion(children=[widgets.VBox([video_button,video_output])],selected_index=None)
            extra.set_title(0,'補助動画の出力（必要な場合だけ開く）');display(extra)
        except Exception as exc:print(type(exc).__name__+': '+str(exc))
        finally:load_button.disabled=False
load_button.on_click(show_episode)
display(choice,joint_choice,load_button,output)
show_episode(None)


## 7.3 斜め前方・上方のカメラ動画を出力

このセルを実行すると、評価2試行それぞれについて両視点のMP4を自動出力します。保存済みデータを使った場合も新規に学習した場合も、操作は同じです。左右に並べた動画を試行ごとのタブで表示し、各動画をダウンロードできます。ファイル名は実行日時・試行・視点・出力日時・識別子を含みます。

カメラの役割と提出済みYouTube動画との関係は、次のとおりです。

| カメラ | 視点・用途 | 動画との関係 |
|---|---|---|
| front：斜め前方 | 方策への画像入力と、搬送全体の撮影 | [提出済みYouTube動画](https://www.youtube.com/watch?v=9SkMPO6dn9I)と同じカメラ視点。7の2倍速動画もこの視点の記録から作成します。7.3ではこのカメラ単独の1倍速動画を出力します。 |
| overhead：上方 | 方策への画像入力と、上から見た配置・搬送の確認 | 斜め前方とは別ファイルの1倍速動画として出力します。 |
| side：真横 | 現在は未実装 | 横方向を独立した視点として撮影するには、新しいカメラの追加が必要です。frontを横方向と呼び替えたり、未撮影の映像を生成したりはしません。 |

7と7.3の斜め前方動画は同じ評価試行の記録を使い、再生速度とテロップが異なります。提出済みYouTube動画と同じ視点でも、今回再評価した場合は別の実行記録です。YouTube動画と同一フレーム・同一時刻になることを意味しません。

動画は同一試行の実際の記録を使用し、5 fpsのシミュレーション時間軸で直前画像を保持して再標本化した1倍速です。7の提出用2倍速映像とは別です。二つの動画プレイヤーは独立しており、同時再生・厳密な撮影同期を意味しません。


In [ ]:
from pathlib import Path
import json, datetime, uuid, subprocess, shutil
import numpy as np
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display, Video

def camera_sources(folder):
    """同一試行内の実記録のみを使用。NPZのtop=front、side=overhead。"""
    found={}
    result_path=folder/'result.json'
    result=json.loads(result_path.read_text()) if result_path.is_file() else {}
    if result.get('mode')=='teacher' and (folder/'episode.npz').is_file():
        with np.load(folder/'episode.npz',allow_pickle=False) as a:
            for name,key in [('front','top'),('overhead','side')]:
                if key in a and 'time' in a:
                    frames=a[key].copy();times=a['time'].copy()
                    found[name]=(times,lambda i,frames=frames:frames[i],'teacher RGB observations')
    for name in ['front','overhead']:
        index_path=folder/'camera_views'/name/'frames.json'
        if index_path.is_file():
            index=json.loads(index_path.read_text())
            pairs=[(float(t),index_path.parent/f'{int(i):05d}.jpg') for t,i in index if (index_path.parent/f'{int(i):05d}.jpg').is_file()]
            if pairs:
                times=np.array([x[0] for x in pairs]);paths=[x[1] for x in pairs]
                def read(i,paths=paths):
                    with Image.open(paths[i]) as im:return np.array(im.convert('RGB'))
                found[name]=(times,read,'recorded camera RGB JPEG')
    if 'front' not in found and (folder/'frames.json').is_file():
        index=json.loads((folder/'frames.json').read_text())
        pairs=[(float(t),folder/'frames'/f'{int(i):05d}.jpg') for t,i in index if (folder/'frames'/f'{int(i):05d}.jpg').is_file()]
        if pairs:
            times=np.array([x[0] for x in pairs]);paths=[x[1] for x in pairs]
            def read(i,paths=paths):
                with Image.open(paths[i]) as im:return np.array(im.convert('RGB'))
            found['front']=(times,read,'recorded front JPEG with original captions')
    return found,result

def encode_view(times,read,target,label):
    times=np.asarray(times,dtype=float)
    if len(times)<2 or not np.isfinite(times).all() or np.any(np.diff(times)<=0):
        raise ValueError('画像時刻が不足、重複、または非単調です。')
    ffmpeg=shutil.which('ffmpeg')
    if not ffmpeg:raise RuntimeError('ffmpegが必要です。Colabで !apt-get -qq install -y ffmpeg を実行してください。')
    temp=target.with_name(target.stem+'_building.mp4')
    with target.with_suffix('.log').open('w') as log:
        proc=subprocess.Popen([ffmpeg,'-y','-loglevel','error','-f','rawvideo','-pix_fmt','rgb24','-s','640x520','-r','5','-i','-',
                               '-an','-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(temp)],stdin=subprocess.PIPE,stderr=log)
        try:
            for t in np.arange(times[0],times[-1]+1e-8,.2):
                i=int(np.clip(np.searchsorted(times,t,side='right')-1,0,len(times)-1))
                frame=Image.fromarray(read(i)).convert('RGB');frame.thumbnail((640,480))
                panel=Image.new('RGB',(640,520),(20,25,35));panel.paste(frame,((640-frame.width)//2,(480-frame.height)//2))
                draw=ImageDraw.Draw(panel);draw.text((10,485),label,fill='white')
                draw.text((10,502),f'1x simulation time | frame t={times[i]:.3f}s',fill='white')
                proc.stdin.write(np.asarray(panel,dtype=np.uint8).tobytes())
            proc.stdin.close()
            if proc.wait():raise RuntimeError('ffmpegが失敗しました: '+str(target.with_suffix('.log')))
        finally:
            if proc.poll() is None:proc.terminate();proc.wait()
    temp.replace(target)

if not both_views_complete(EVAL/'evaluation'):
    raise RuntimeError('両視点の評価記録が揃っていません。6の結果を確認してください。')
token=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'_'+uuid.uuid4().hex[:8]
VIS_RUN=RUN
VIS_OUT=RUN/'visualization'/('camera_videos_'+token)
VIS_OUT.mkdir(parents=True,exist_ok=False)
outputs=[];panels=[]
for seed in [900,901]:
    folder=EVAL/'evaluation'/f'hybrid_{seed}'
    sources,result=camera_sources(folder)
    if not all(name in sources for name in ['front','overhead']):
        raise RuntimeError('同じ試行の両視点が必要です: '+str(folder))
    boxes=[]
    for name in ['front','overhead']:
        target=VIS_OUT/f'{RUN.name}_seed{seed}_{name}_{token}_1x.mp4'
        times,read,origin=sources[name]
        print(f'動画出力中: seed {seed} / {name}',flush=True)
        encode_view(times,read,target,f'{name} | seed={seed} | policy | success={result.get("success")}')
        outputs.append(dict(seed=seed,view=name,file=target.name,source=origin))
        output=widgets.Output()
        with output:display(Video(str(target),embed=True,width=400))
        download=widgets.Button(description=name+'動画をダウンロード',layout=widgets.Layout(width='240px'))
        def get_file(_,target=target):
            from google.colab import files
            files.download(str(target))
        download.on_click(get_file)
        boxes.append(widgets.VBox([widgets.Label(name),output,download]))
    panels.append(widgets.HBox(boxes,layout=widgets.Layout(overflow='auto')))
tabs=widgets.Tab(children=panels)
for i,seed in enumerate([900,901]):tabs.set_title(i,f'seed {seed}')
display(tabs)
(VIS_OUT/'manifest.json').write_text(json.dumps(dict(outputs=outputs,playback='1x simulation time; 5 fps latest-frame hold',
    note='Same policy trial per pair; asynchronous camera streams; no synthetic images.'),indent=2))
print('両試行×両視点の4動画を保存しました:',VIS_OUT)
print('続いて8を実行してください。')


## 8 結果とモデルをDriveへ保存

Google Driveに保存したデータを使用する場合も、新規にデータ収集・学習を行う場合も、最後にこのセルを実行し、今回の成果物をGoogle Driveへ保存してください。新しい日時の作業フォルダを丸ごと保存します。復元した元のDriveフォルダは上書きしません。7の動画、7.2の画像・CSV、7.3の両視点動画も保存対象です。モデルが含まれない記録だけを再利用した場合は、その出所を`run_origin.json`に残します。

Driveの認証・接続エラー時は、状態を確認してこのセルを個別に再実行してください。保存先が既にある場合は、新しいバックアップ名で保存します。

Driveへの接続・認証でエラーになった場合は、ランタイムへの接続を確認し、この8のセルだけを再実行してください。認証画面が表示されたらアクセスを許可します。`Not connected to a runtime`の場合は、先にColab右上の「接続」でランタイムへ接続してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if not (VIS_OUT/'manifest.json').is_file():
    raise RuntimeError('7.3の両視点動画出力を完了してください。')
manifest=json.loads((VIS_OUT/'manifest.json').read_text())
if {(x['seed'],x['view']) for x in manifest['outputs']}!={(seed,view) for seed in [900,901] for view in ['front','overhead']}:
    raise RuntimeError('4動画が揃っていません。')
if not all((VIS_OUT/x['file']).is_file() for x in manifest['outputs']):raise RuntimeError('動画ファイルが不足しています。')
dst=Path('/content/drive/MyDrive/PAI_results')/RUN.name
if dst.exists():dst=dst.with_name(RUN.name+'_backup_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
print('保存中:',dst,flush=True)
shutil.copytree(RUN,dst)
print('保存完了:',dst)
